# T6: Dictionaries and Sets
CHE-226, Programming and Data Science

**Learning objective:** represent process data as key-value pairs and unique collections, look up, update, and iterate over them, build them with comprehensions, and use set operations to compare streams.

**Teaches:** (KA-01)

*Spans two class meetings; see the class break marker partway through.*

## Agenda: This Class
1. Creating a dictionary
2. Updating, removing, and testing keys
3. Iterating: keys, values, and items
4. Comparing and merging dictionaries
5. Counting with dictionaries

# 1. Creating a Dictionary

### Concept
A dictionary maps unique keys to values: `{key: value, ...}`. You look a value up by its key, not by a position. Keys must be immutable (strings, numbers, tuples).

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
feed = {"water": 0.70, "ethanol": 0.25, "methanol": 0.05}  # mass fr.

print(feed["ethanol"])         # look up by key
print(len(feed))               # number of key-value pairs
print(feed)

Detailed notes: In T5 a composition needed two parallel lists (component names and fractions) kept in the same order; a dictionary stores the pairing itself, so feed['ethanol'] reads like the engineering question. Lookups by key are fast regardless of size, because dictionaries are hash tables. Since Python 3.7, dictionaries remember insertion order, but they are still not indexed by position: feed[0] looks for a key equal to 0 and raises KeyError. Keys must be hashable, which in practice means immutable; a tuple such as ('S1', 'water') is a valid key, a list is not.

### Activity: predict then run
The key `"water"` appears twice. How many entries does this dictionary have, and what is the water fraction?

In [ ]:
stream = {"water": 0.70, "ethanol": 0.30, "water": 0.60}
print(len(stream), stream["water"])

Answer: 2 0.6. Keys are unique, so the second 'water' overwrites the first; Python gives no warning. Note that the fractions no longer sum to 1, a silent data-entry error worth pointing out.

# 2. Updating, Removing, and Testing Keys

### Concept
Assigning to a key updates it, or adds it if new. `del` and `pop` remove a key. `in` tests for a key; `get` returns a default instead of raising `KeyError`.

In [ ]:
feed = {"water": 0.70, "ethanol": 0.25, "methanol": 0.05}
feed["ethanol"] = 0.28              # update an existing key
feed["butanol"] = 0.00              # add a new key
del feed["butanol"]                 # remove it again
print("methanol" in feed)           # key test
print(feed.get("acetone", 0.0))     # missing key, default 0.0

Detailed notes: d[key] = value never fails: it updates or inserts. d[key] for reading raises KeyError when the key is missing, so there are two safe patterns: test with in first, or use d.get(key, default). pop(key) removes and returns the value, which is handy when moving an entry from one dictionary to another. The in operator checks keys only; to search values, use value in d.values(). That distinction is the bug in the activity.

### Activity: spot the bug
This should report that some component is at 0.25. It runs, but prints the wrong message. Why?

In [ ]:
feed = {"water": 0.70, "ethanol": 0.25, "methanol": 0.05}
if 0.25 in feed:
    print("A component is at 25 percent")
else:
    print("No component at 25 percent")

Answer: prints 'No component at 25 percent'. in on a dictionary tests keys, and 0.25 is a value. Fix: `if 0.25 in feed.values():`.

# 3. Iterating: keys, values, and items

### Concept
A `for` loop over a dictionary visits its keys. `.values()` gives the values and `.items()` gives `(key, value)` pairs, ready to unpack.

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
feed = {"water": 0.60, "ethanol": 0.30, "methanol": 0.10}

for comp, x in feed.items():            # unpack each pair
    print(f"{comp:<9} {x:.2f}")
print(sorted(feed))                     # keys, alphabetical
print(max(feed, key=feed.get))          # key with largest value

Detailed notes: keys, values, and items return views: live windows onto the dictionary that reflect later changes, rather than copies; wrap them in list() to take a snapshot. Iterating over items and unpacking is the most common idiom and avoids a second lookup inside the loop. sorted(d) sorts the keys; max(d, key=d.get) compares keys by their values, the dictionary version of the key= trick from T5. Do not add or remove keys while looping over a dictionary; Python raises RuntimeError. The activity is a floating-point surprise worth making explicit once: mass fractions entered as decimals rarely sum to exactly 1.

### Activity: predict then run
These fractions obviously add to 1. What does the comparison print?

In [ ]:
total = sum(feed.values())
print(total == 1.0)
print(total)

Answer: False, then 0.9999999999999999. 0.6, 0.3, and 0.1 cannot be stored exactly in binary floating point, so the sum is off in the last digit. Compare with a tolerance instead: math.isclose(total, 1.0) or abs(total - 1) < 1e-9. This matters every time a program checks a material balance.

# 4. Comparing and Merging Dictionaries

### Concept
`==` is True when two dictionaries hold the same key-value pairs, in any order. `update` merges another dictionary in, overwriting shared keys.

In [ ]:
lab = {"T_K": 350.0, "P_bar": 5.0}
dcs = {"P_bar": 5.0, "T_K": 350.0}      # same data, different order
print(lab == dcs)                       # order does not matter

lab.update({"P_bar": 5.2, "flow_kg_h": 120.0})
print(lab)

Detailed notes: Equality compares contents, not insertion order or identity. update(other) walks other's pairs: existing keys take the new value, new keys are added; nothing is returned (it mutates in place, like list.sort). Python 3.9+ also offers merged = a | b, which builds a new dictionary without changing either input, with b winning on conflicts. Merging is the everyday operation behind combining a design specification with plant measurements, where the measured values should override the design ones.

### Activity: predict then run
Which value of `T_K` survives each merge?

In [ ]:
design = {"T_K": 350.0, "P_bar": 5.0}
measured = {"T_K": 347.5}
print((design | measured)["T_K"], (measured | design)["T_K"])

Answer: 347.5 350.0. With |, the right-hand dictionary wins shared keys, so the order of the operands decides which temperature is kept.

# 5. Counting with Dictionaries

### Concept
A dictionary makes a natural tally: key = thing seen, value = how many times. `collections.Counter` does the same in one line.

In [ ]:
alarms = ["TAH-101", "PAL-205", "TAH-101", "LAH-310", "TAH-101",
          "PAL-205"]                    # alarm tags from a shift log

counts = {}
for tag in alarms:
    counts[tag] = counts.get(tag, 0) + 1    # 0 the first time
print(counts)

from collections import Counter
print(Counter(alarms).most_common(1))

Detailed notes: counts.get(tag, 0) + 1 is the counting idiom: the first time a tag appears there is no entry, so get supplies 0. Without get, the first occurrence raises KeyError. Counter is a dictionary subclass built for exactly this; most_common(n) returns the n most frequent (key, count) pairs. The same pattern counts word frequencies in text, which is the classic textbook example; alarm tags are the plant equivalent, and alarm floods, where a few tags dominate a log, are a real operability problem.

### Activity: trace the variable
Trace `counts` through this short log. What is printed?

In [ ]:
log = ["FAL-1", "FAL-1", "TAH-2", "FAL-1"]
counts = {}
for tag in log:
    counts[tag] = counts.get(tag, 0) + 1
print(counts, len(counts))

Answer: {'FAL-1': 3, 'TAH-2': 1} 2. len counts distinct tags, not total alarms; sum(counts.values()) would give 4.

## Class Break
Covered so far: creating, updating, and iterating dictionaries, comparing and merging them, and counting with get and Counter.

## Agenda: Next Class
6. Dictionary comprehensions
7. Nested dictionaries: a flowsheet
8. Sets: unique collections
9. Set operations
10. Putting it together: a mixer component balance

# 6. Dictionary Comprehensions

### Concept
`{k: expr for k, v in d.items()}` builds a new dictionary in one line, the dictionary version of a list comprehension.

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
mass_kg = {"water": 36.0, "ethanol": 46.0}     # component masses
MW = {"water": 18.0, "ethanol": 46.0}          # g/mol

moles = {c: m / MW[c] for c, m in mass_kg.items()}   # kmol
total = sum(moles.values())
x = {c: n / total for c, n in moles.items()}         # mole fractions
print(moles)
print({c: round(v, 3) for c, v in x.items()})

Detailed notes: This is T1's mole-fraction mini-challenge done with dictionaries: matching MW[c] by component name removes any chance of pairing a mass with the wrong molar mass, which parallel lists cannot guarantee. A comprehension can also filter ({c: v for c, v in d.items() if v > 0.1}) or swap keys and values ({v: k for k, v in d.items()}), which only works if the values are unique and hashable. Units follow the data: kg divided by g/mol gives kmol, but the mole fractions are dimensionless either way.

### Activity: predict then run
Masses are 36 kg water and 46 kg ethanol. Is the water mole fraction above or below 0.5? What does this print?

In [ ]:
print(x["water"] > 0.5, round(x["water"], 3))

Answer: True 0.667. Water has less mass but a much smaller molar mass, so 2 kmol of water versus 1 kmol of ethanol gives 2/3. Same lesson as T1: mass does not decide mole fraction.

# 7. Nested Dictionaries: A Flowsheet

### Concept
A dictionary's values can be dictionaries. One outer key per stream, one inner dictionary per composition, gives a small flowsheet in plain Python.

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
flowsheet = {
    "S1": {"flow_kg_h": 100.0, "w": {"water": 0.7, "ethanol": 0.3}},
    "S2": {"flow_kg_h": 60.0, "w": {"water": 0.2, "ethanol": 0.8}},
}
print(flowsheet["S2"]["w"]["ethanol"])        # chain the lookups
for name, s in flowsheet.items():
    m_EtOH = s["flow_kg_h"] * s["w"]["ethanol"]
    print(f"{name}: {m_EtOH:.0f} kg/h ethanol")

Detailed notes: Each [ ] step goes one level deeper; reading the chain left to right (stream, then composition, then component) mirrors how an engineer reads a stream table. Nested dictionaries are also exactly the shape of JSON, which T9 saves to and loads from files. The trade-off against a 2D list: dictionaries label everything and tolerate missing entries (a stream with no methanol simply has no key), while lists are compact but rely on remembering column order. When the table gets large and regular, a pandas DataFrame (T9) combines both advantages.

### Activity: cold call
Why is a dictionary a better fit than a list for storing these streams? One sentence.

Answer: streams are identified by name, not position, so flowsheet['S2'] is unambiguous even if streams are added or reordered, and each stream can hold a different set of components. A strong answer mentions lookup by meaningful key; a weak one says 'it is faster' without saying why that matters here.

# 8. Sets: Unique Collections

### Concept
A set `{a, b, c}` holds unique, unordered, immutable items. Building a set from a list drops duplicates; `in` tests membership quickly.

In [ ]:
seen = ["water", "ethanol", "water", "methanol", "ethanol"]
species = set(seen)                   # duplicates removed
print(species, len(species))
print("acetone" in species)

species.add("acetone")                # sets are mutable
print(sorted(species))                # sorted() gives a list

Detailed notes: Sets are unordered, so printing may show a different order each session, and indexing (species[0]) raises TypeError; sort into a list when order matters. Items must be hashable, like dictionary keys; a set is essentially a dictionary with keys and no values. The most common beginner trap, used in the activity: {} is an empty dictionary, not an empty set, so an empty set must be written set(). frozenset is the immutable variant, usable as a dictionary key or inside another set.

### Activity: predict then run
What types and lengths do these print?

In [ ]:
a = {}
b = set()
c = {"H2O", "H2O", "CO2"}
print(type(a), type(b), len(c))

Answer: <class 'dict'> <class 'set'> 2. Empty braces make a dictionary; duplicates collapse in a set, so c has 2 items.

# 9. Set Operations

### Concept
`|` union, `&` intersection, `-` difference, `^` symmetric difference. `<=` tests subset. They answer comparison questions about two collections.

In [ ]:
feed = {"water", "ethanol", "methanol", "acetone"}
product = {"ethanol", "water"}

print(feed & product)       # in both streams
print(feed - product)       # in feed but not product: removed
print(product <= feed)      # is product a subset of feed?
print(feed | {"CO2"})       # union: everything seen

Detailed notes: These operators are Venn diagrams in code. Each also has a method form (union, intersection, difference, symmetric_difference, issubset, issuperset), and the method forms accept any iterable, not just sets. Mutating versions (|=, &=, -=, and add, discard, remove) change the set in place; discard ignores a missing item while remove raises KeyError. Set comprehensions, {x for x in ...}, work like list comprehensions. A typical plant use: which species appear in the feed but not in any product stream, and so must leave via a purge or waste stream?

### Activity: predict then run
Using `feed` and `product` above, what does each line print?

In [ ]:
print(product ^ feed == feed - product)
print(len(feed | product), product.isdisjoint({"CO2"}))

Answer: True, then 4 True. product is a subset of feed, so the symmetric difference equals feed minus product (methanol and acetone). The union still has 4 items because nothing new is added, and product shares nothing with {'CO2'}.

# 10. Putting It Together: A Mixer Component Balance

### Concept
Two streams enter a mixer. A set union collects every component; `get` with a default of 0 handles components missing from one stream; a dict comprehension gives the outlet.

*Ties to T-03.01.02 Composition Measures (KA-03)*

In [ ]:
s1 = {"water": 70.0, "ethanol": 30.0}                    # kg/h
s2 = {"water": 10.0, "ethanol": 40.0, "methanol": 10.0}  # kg/h

comps = set(s1) | set(s2)                    # every component
out = {c: s1.get(c, 0) + s2.get(c, 0) for c in sorted(comps)}
total = sum(out.values())
w = {c: round(m / total, 3) for c, m in out.items()}
print(out, total)
print(w)

Detailed notes: This combines the lecture: set(dict) gives a dictionary's keys as a set, a union finds every component entering, get with a default of 0 treats an absent component as zero flow, and two comprehensions give the outlet flows and mass fractions. It is a component material balance on a mixer (in = out, no reaction), the first calculation in any mass and energy balance course, written so that adding a third inlet or a new species needs no change to the logic.

### Activity: cold call
If a third inlet of pure methanol at 20 kg/h joined the mixer, what would the new total be, and which fraction would change the most?

In [ ]:
s3 = {"methanol": 20.0}
comps = set(s1) | set(s2) | set(s3)
out = {c: s1.get(c, 0) + s2.get(c, 0) + s3.get(c, 0) for c in comps}
print(sum(out.values()), round(out["methanol"] / 180, 3))

Answer: total rises from 160 to 180 kg/h; methanol changes most, from 0.062 to 0.167, while water and ethanol fall because the total grew. Good check that students see every fraction change when one flow changes.

## Recap and Lab Practice
- A dictionary maps unique, immutable keys to values; get and in handle missing keys; items() gives pairs to unpack.
- Comprehensions and nested dictionaries model compositions and flowsheets by name rather than position.
- A set holds unique items; union, intersection, and difference compare collections. {} is a dict, set() is an empty set.

Lab practice: store a four-stream flowsheet as nested dictionaries, check each stream's fractions sum to 1 with a tolerance, and write a general mixer function for any number of inlet streams; ask if you want them turned into a lab worksheet.

Next lecture: T7, array-oriented programming with NumPy.